# Debug — EndoSfMLearner baseline (split oficial endovis)

Notebook minimo para aislar **solo EndoSfMLearner** y reproducir su *baseline* de referencia (AbsRel ~0.070).

Replica el protocolo del `evaluate_depth.py` oficial de AF-SfMLearner (el que el profe usa para todos los modelos):
- Split oficial `splits/endovis/test_files.txt` (550 frames, datasets 1-7)
- GT identico a `export_gt_depth.py`: canal Z del `scene_points{N-1}.tiff`, crop `[0:1024, :]`
- `MIN_DEPTH=1e-3`, `MAX_DEPTH=150`, median scaling
- Inferencia con el `DispResNet` nativo de EndoSLAM (256x832, norm 0.45/0.225, `depth = 1/disp`)

In [ ]:
from pathlib import Path
import sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if not IN_COLAB:
    try:
        import google.colab
        IN_COLAB = True
    except ImportError:
        IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    BASE         = Path("/content/drive/MyDrive/proyecto_integrador")
    SCARED_ROOT  = BASE / "scared_raw"
    ENDOSLAM_PATH = BASE / "EndoSLAM"
    W            = BASE / "scared weights"
    REPO_ROOT    = Path("/content/repo_52")
    if not REPO_ROOT.exists():
        subprocess.check_call(["git","clone","--depth=1",
            "https://github.com/jmtoral/proyecto_integrador_52.git", str(REPO_ROOT)])
    else:
        subprocess.check_call(["git","-C",str(REPO_ROOT),"fetch","origin"])
        subprocess.check_call(["git","-C",str(REPO_ROOT),"reset","--hard","origin/main"])
    SPLIT_FILE = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"
else:
    BASE          = Path("E:/scared_wights_complete/scared weights")
    SCARED_ROOT   = Path("D:/Proyecto_Integrador/Corrreccion_Luz/data/scared_raw")
    ENDOSLAM_PATH = Path("E:/EndoSLAM")
    W             = BASE
    REPO_ROOT     = Path(r"d:\Proyecto_Integrador\Corrreccion_Luz")
    SPLIT_FILE    = REPO_ROOT / "data" / "splits" / "endovis" / "test_files.txt"

W_ENDOSFM       = W / "endosfmlearner_weights" / "11-09-03_58"
ENDOSFM_WEIGHTS = W_ENDOSFM / "dispnet_model_best.pth.tar"
NPZ_CACHE       = BASE / "split_frames.npz"
CAP_MM = 150.0

def load_split(split_file):
    items = []
    with open(split_file) as f:
        for line in f:
            line = line.strip()
            if not line: continue
            folder, frame_id, _side = line.split()
            ds, kf = folder.split("/")
            ds_n = "dataset_" + ds.replace("dataset","")
            kf_n = "keyframe_" + kf.replace("keyframe","")
            items.append((ds_n, kf_n, int(frame_id)))
    return items

SPLIT_ITEMS = load_split(SPLIT_FILE)
print(f"Entorno : {'Colab' if IN_COLAB else 'Local'}")
print(f"Split   : {len(SPLIT_ITEMS)} frames (datasets {sorted({d for d,_,_ in SPLIT_ITEMS})})")
print(f"Pesos   : {'OK' if ENDOSFM_WEIGHTS.exists() else 'NO ENCONTRADO'}  {ENDOSFM_WEIGHTS}")
print(f"NPZ GT  : {'OK' if NPZ_CACHE.exists() else 'NO ENCONTRADO'}  {NPZ_CACHE}")

In [ ]:
import torch, importlib.util

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", DEVICE)

_endosfm_dir = ENDOSLAM_PATH / "EndoSfMLearner"
if str(_endosfm_dir) not in sys.path:
    sys.path.insert(0, str(_endosfm_dir))

_spec = importlib.util.spec_from_file_location(
    "_endosfm_models_dbg",
    str(_endosfm_dir / "models" / "__init__.py"),
    submodule_search_locations=[str(_endosfm_dir / "models")])
endosfm_models = importlib.util.module_from_spec(_spec)
sys.modules["_endosfm_models_dbg"] = endosfm_models
_spec.loader.exec_module(endosfm_models)

endosfm = endosfm_models.DispResNet(18, False).to(DEVICE)
w = torch.load(ENDOSFM_WEIGHTS, map_location=DEVICE)
endosfm.load_state_dict(w["state_dict"])
endosfm.eval()
print(f"EndoSfMLearner cargado: {sum(p.numel() for p in endosfm.parameters())/1e6:.2f} M params")

In [ ]:
import numpy as np

# Cargar imagenes + GT desde el npz cacheado (mismo que el notebook principal).
# El GT ya viene como canal Z del tiff, crop [0:1024,:], igual que export_gt_depth.py oficial.
def _key(ds, kf, fid): return f"{ds}|{kf}|{fid}"

SPLIT_DATA = {}
assert NPZ_CACHE.exists(), "No existe split_frames.npz — corre primero el notebook principal para generarlo"
_npz = np.load(NPZ_CACHE, allow_pickle=True)
for ds, kf, fid in SPLIT_ITEMS:
    k = _key(ds, kf, fid)
    ik, gk = "img_"+k, "gt_"+k
    if ik in _npz.files:
        gt = _npz[gk] if gk in _npz.files else None
        if gt is not None and gt.size == 1 and np.isnan(gt).all(): gt = None
        SPLIT_DATA[k] = (_npz[ik], gt)
print(f"Frames cargados: {len(SPLIT_DATA)}")
_n_gt = sum(1 for v in SPLIT_DATA.values() if v[1] is not None)
print(f"Con GT valido  : {_n_gt}")

In [ ]:
from skimage.transform import resize as imresize

def predict_endosfm(img):
    """DispResNet nativo: 256x832, norm (x/255-0.45)/0.225, depth = 1/disp (test_disp.py oficial).

    CRITICO: convertir a float ANTES del imresize. skimage.resize sobre uint8 normaliza a [0,1];
    sobre float preserva el rango 0-255 (como hace imread().astype(float32) en test_disp.py).
    El bug previo (resize sobre uint8 -> /255) dejaba la imagen en ~0-0.004 -> prediccion mala."""
    H, W_ = img.shape[:2]
    r = imresize(img.astype(np.float32), (256, 832)).astype(np.float32)  # float ANTES de resize
    t = torch.from_numpy(((r/255-0.45)/0.225).transpose(2,0,1)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        d = endosfm(t)
    pred_depth = 1.0 / (d.squeeze().detach().cpu().numpy() + 1e-6)
    return imresize(pred_depth, (H, W_))

In [ ]:
# Evaluacion estilo evaluate_depth.py oficial: median scaling, MIN_DEPTH=1e-3, MAX_DEPTH=150
from collections import defaultdict
from tqdm import tqdm

MIN_DEPTH, MAX_DEPTH = 1e-3, 150.0

def compute_errors(gt, pred):
    thresh = np.maximum((gt/pred), (pred/gt))
    a1 = (thresh < 1.25).mean(); a2 = (thresh < 1.25**2).mean(); a3 = (thresh < 1.25**3).mean()
    rmse = np.sqrt(((gt-pred)**2).mean())
    rmse_log = np.sqrt(((np.log(gt)-np.log(pred))**2).mean())
    abs_rel = np.mean(np.abs(gt-pred)/gt)
    sq_rel = np.mean(((gt-pred)**2)/gt)
    return abs_rel, sq_rel, rmse, rmse_log, a1, a2, a3

errors = []; ratios = []; per_ds = defaultdict(list)
for ds, kf, fid in tqdm(SPLIT_ITEMS, desc="EndoSfMLearner"):
    k = _key(ds, kf, fid)
    if k not in SPLIT_DATA: continue
    img, gt = SPLIT_DATA[k]
    if gt is None: continue
    pred = predict_endosfm(img)
    # mask oficial: gt > MIN_DEPTH & gt < MAX_DEPTH (sin crop extra en endovis)
    mask = (gt > MIN_DEPTH) & (gt < MAX_DEPTH) & (~np.isnan(gt))
    if mask.sum() == 0: continue
    pred_m = pred[mask]; gt_m = gt[mask]
    ratio = np.median(gt_m) / np.median(pred_m)
    ratios.append(ratio)
    pred_m = pred_m * ratio
    pred_m[pred_m < MIN_DEPTH] = MIN_DEPTH
    pred_m[pred_m > MAX_DEPTH] = MAX_DEPTH
    e = compute_errors(gt_m, pred_m)
    errors.append(e); per_ds[ds].append(e[0])

mean_errors = np.array(errors).mean(0)
print(f"\nEvaluados: {len(errors)} frames")
print(f"Scaling ratio mediano: {np.median(ratios):.2f}\n")
print(f"{'abs_rel':>9}{'sq_rel':>9}{'rmse':>9}{'rmse_log':>10}{'a1':>8}{'a2':>8}{'a3':>8}")
print(("{:9.4f}"*4 + "{:8.4f}"*3).format(*mean_errors))
print(f"\n>>> AbsRel global EndoSfMLearner: {mean_errors[0]:.4f}  (referencia profe: ~0.0700)\n")
print("AbsRel por dataset:")
for ds in sorted(per_ds):
    print(f"  {ds}: {np.mean(per_ds[ds]):.4f}  (n={len(per_ds[ds])})")

---
## MonoViT — diagnostico

Carga MonoViT (encoder mpvit_small + decoder detectado), reporta el tipo de decoder y evalua con el mismo protocolo.

In [ ]:
import importlib.util as _ilu, types
import torch.nn as nn
from collections import OrderedDict

W_MONOVIT = W / "monovit_weights" / "weights_19"
MONOVIT_PATH = (BASE / "MonoViT") if IN_COLAB else Path("E:/MonoViT")
_NETDIR = MONOVIT_PATH / "networks"

for _k in list(sys.modules):
    if _k == "networks" or _k.startswith("networks."):
        del sys.modules[_k]
_pkg = types.ModuleType("networks"); _pkg.__path__ = [str(_NETDIR)]
sys.modules["networks"] = _pkg
def _load_sub(name, fname):
    spec = _ilu.spec_from_file_location(f"networks.{name}", str(_NETDIR / fname))
    m = _ilu.module_from_spec(spec); sys.modules[f"networks.{name}"] = m
    spec.loader.exec_module(m); setattr(_pkg, name, m); return m
_load_sub("hr_layers", "hr_layers.py")
mpvit_small = _load_sub("mpvit", "mpvit.py").mpvit_small

# DepthDecoder simple (Monodepth2-style) con num_ch_dec = num_ch_enc del mpvit.
# El checkpoint de MonoViT del profe usa ESTE decoder (keys decoder.N, sin convs.f / X_00).
def _up(x): return nn.functional.interpolate(x, scale_factor=2, mode="nearest")
class _Conv3x3(nn.Module):
    def __init__(s,i,o): super().__init__(); s.pad=nn.ReflectionPad2d(1); s.conv=nn.Conv2d(int(i),int(o),3)
    def forward(s,x): return s.conv(s.pad(x))
class _ConvBlock(nn.Module):
    def __init__(s,i,o): super().__init__(); s.conv=_Conv3x3(i,o); s.nl=nn.ELU(inplace=False)
    def forward(s,x): return s.nl(s.conv(x))
class MonoViTDepthDecoderSimple(nn.Module):
    def __init__(s, num_ch_enc=[64,128,216,288,288], scales=range(4), use_skips=True):
        super().__init__(); s.scales=list(scales); s.use_skips=use_skips
        s.num_ch_enc=np.array(num_ch_enc); s.num_ch_dec=np.array(num_ch_enc); s.convs=OrderedDict()
        for i in range(4,-1,-1):
            ci=s.num_ch_enc[-1] if i==4 else s.num_ch_dec[i+1]
            s.convs[("upconv",i,0)]=_ConvBlock(ci,s.num_ch_dec[i]); ci=s.num_ch_dec[i]
            if s.use_skips and i>0: ci+=s.num_ch_enc[i-1]
            s.convs[("upconv",i,1)]=_ConvBlock(ci,s.num_ch_dec[i])
        for sc in s.scales: s.convs[("dispconv",sc)]=_Conv3x3(s.num_ch_dec[sc],1)
        s.decoder=nn.ModuleList(list(s.convs.values())); s.sigmoid=nn.Sigmoid()
    def forward(s, feats):
        out={}; x=feats[-1]
        for i in range(4,-1,-1):
            x=s.convs[("upconv",i,0)](x); x=[_up(x)]
            if s.use_skips and i>0: x+=[feats[i-1]]
            x=torch.cat(x,1); x=s.convs[("upconv",i,1)](x)
            if i in s.scales: out[("disp",i)]=s.sigmoid(s.convs[("dispconv",i)](x))
        return out

mvenc = mpvit_small(); mvenc.num_ch_enc = [64,128,216,288,288]
_ed = torch.load(W_MONOVIT / "encoder.pth", map_location=DEVICE)
MV_H = _ed.get("height",192); MV_W = _ed.get("width",640)
mvenc.load_state_dict({k:v for k,v in _ed.items() if k in mvenc.state_dict()})
mvenc.to(DEVICE).eval()

_sd = torch.load(W_MONOVIT / "depth.pth", map_location=DEVICE)
mvdec = MonoViTDepthDecoderSimple([64,128,216,288,288])
_res = mvdec.load_state_dict(_sd, strict=False)
print("missing:", len(_res.missing_keys), "unexpected:", len(_res.unexpected_keys))
mvdec.to(DEVICE).eval()
print(f"MonoViT cargado (decoder simple): {MV_H}x{MV_W}")

In [ ]:
import PIL.Image as pil, cv2
from torchvision import transforms

def predict_monovit(img, max_depth=150.0, min_depth=0.1):
    H, W_ = img.shape[:2]
    t = transforms.ToTensor()(pil.fromarray(img).resize((MV_W,MV_H),pil.LANCZOS)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = mvdec(mvenc(t))
    disp = out[("disp",0)].squeeze().detach().cpu().numpy()
    min_disp, max_disp = 1.0/max_depth, 1.0/min_depth
    sd = min_disp + (max_disp-min_disp)*disp
    sd = cv2.resize(sd, (W_, H))
    return 1.0/sd

errors=[]; ratios=[]; per_ds=defaultdict(list)
for ds, kf, fid in tqdm(SPLIT_ITEMS, desc="MonoViT"):
    k=_key(ds,kf,fid)
    if k not in SPLIT_DATA: continue
    img,gt=SPLIT_DATA[k]
    if gt is None: continue
    pred=predict_monovit(img)
    mask=(gt>MIN_DEPTH)&(gt<MAX_DEPTH)&(~np.isnan(gt))
    if mask.sum()==0: continue
    pm=pred[mask]; gm=gt[mask]
    r=np.median(gm)/np.median(pm); ratios.append(r); pm=pm*r
    pm[pm<MIN_DEPTH]=MIN_DEPTH; pm[pm>MAX_DEPTH]=MAX_DEPTH
    e=compute_errors(gm,pm); errors.append(e); per_ds[ds].append(e[0])

me=np.array(errors).mean(0)
print(f"\nEvaluados: {len(errors)} | ratio mediano: {np.median(ratios):.2f}")
print(f">>> AbsRel global MonoViT: {me[0]:.4f}  (referencia profe: ~0.0790)\n")
for ds in sorted(per_ds):
    print(f"  {ds}: {np.mean(per_ds[ds]):.4f}  (n={len(per_ds[ds])})")

In [ ]:
# weights_19_MonoViT: ESTE es el MonoViT OFICIAL real (decoder HR-Depth: convs.f4, X_00, attention).
# Se carga EXACTAMENTE como evaluate_depth.py oficial: networks.DepthDecoder() sin args.
W_W19 = W / "weights_19_MonoViT" / "weights_19"

# Cargar el DepthDecoder HR oficial del repo MonoViT
_hr = _load_sub("hr_decoder", "hr_decoder.py")
DepthDecoderHR = _hr.DepthDecoder

w19enc = mpvit_small(); w19enc.num_ch_enc = [64,128,216,288,288]
_ed19 = torch.load(W_W19 / "encoder.pth", map_location=DEVICE)
W19_H = _ed19.get("height",192); W19_W = _ed19.get("width",640)
w19enc.load_state_dict({k:v for k,v in _ed19.items() if k in w19enc.state_dict()})
w19enc.to(DEVICE).eval()

_sd19 = torch.load(W_W19 / "depth.pth", map_location=DEVICE)
w19dec = DepthDecoderHR()  # defaults oficiales (igual que evaluate_depth.py)
_res19 = w19dec.load_state_dict(_sd19, strict=False)
print("w19 HR -> missing:", len(_res19.missing_keys), "unexpected:", len(_res19.unexpected_keys), f"| {W19_H}x{W19_W}")
w19dec.to(DEVICE).eval()

def predict_w19(img, max_depth=150.0, min_depth=0.1):
    H, W_ = img.shape[:2]
    t = transforms.ToTensor()(pil.fromarray(img).resize((W19_W,W19_H),pil.LANCZOS)).unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        out = w19dec(w19enc(t))
    disp = out[("disp",0)].squeeze().detach().cpu().numpy()
    min_disp, max_disp = 1.0/max_depth, 1.0/min_depth
    sd = min_disp + (max_disp-min_disp)*disp
    sd = cv2.resize(sd, (W_, H))
    return 1.0/sd

errors=[]; ratios=[]; per_ds=defaultdict(list)
for ds, kf, fid in tqdm(SPLIT_ITEMS, desc="w19-MonoViT (HR oficial)"):
    k=_key(ds,kf,fid)
    if k not in SPLIT_DATA: continue
    img,gt=SPLIT_DATA[k]
    if gt is None: continue
    pred=predict_w19(img)
    mask=(gt>MIN_DEPTH)&(gt<MAX_DEPTH)&(~np.isnan(gt))
    if mask.sum()==0: continue
    pm=pred[mask]; gm=gt[mask]
    r=np.median(gm)/np.median(pm); ratios.append(r); pm=pm*r
    pm[pm<MIN_DEPTH]=MIN_DEPTH; pm[pm>MAX_DEPTH]=MAX_DEPTH
    e=compute_errors(gm,pm); errors.append(e); per_ds[ds].append(e[0])

me=np.array(errors).mean(0)
print(f"\nEvaluados: {len(errors)} | ratio mediano: {np.median(ratios):.2f}")
print(f">>> AbsRel global w19-MonoViT (HR oficial): {me[0]:.4f}  (referencia MonoViT profe: ~0.0790)\n")
for ds in sorted(per_ds):
    print(f"  {ds}: {np.mean(per_ds[ds]):.4f}  (n={len(per_ds[ds])})")